# Programmable Data Planes — Homework Report

**Author:** _Your name_  
**Date:** _YYYY-MM-DD_

This notebook summarizes **Task I** (software MAWI analysis), **Task II** (P4/BMv2 in-network extraction), and **Task III** (comparison and discussion).

> Fill in each `TODO` section with your own observations, figures, and numbers.

## Setup

Run from the repo root (or inside Docker with `/repo` and `/p4` mounts). Adjust `N` if you used a different sample size.

In [ ]:
from pathlib import Path

REPO = Path("..").resolve()  # repo root when cwd=notebooks/
N = 10_000

TASK1_RESULTS = REPO / "task1" / "results" / f"n_{N}"
TASK2_RESULTS = REPO / "task2" / "results" / f"n_{N}"

TASK1_JSONL = REPO / "201302011400.jsonl"  # Task I export
TASK2_CAPTURE = REPO / "task2" / "data" / "capture.jsonl"  # Task II telemetry

TASK1_PLOTS = TASK1_RESULTS / "plots"
TASK2_PLOTS = TASK2_RESULTS / "plots"
TASK1_FITS = TASK1_RESULTS / "statistical-fits" / "best_fits_summary.csv"
TASK2_FITS = TASK2_RESULTS / "statistical-fits" / "best_fits_summary.csv"

print("Task I plots:", TASK1_PLOTS)
print("Task II plots:", TASK2_PLOTS)

---
## Task I — Software MAWI analysis (brief recap)

**Data source:** `201302011400.dump.gz` → JSONL via `task1/src/extraction/`

**Metrics:** IAT, frame length (L2), flow duration/size/throughput, segmented by transport, control/data, mice/elephant, app protocol.

### TODO: Key Task I findings

- Sample size: _N = ?_
- Dominant transport / apps:
- Notable distribution shapes (from fits):
- Example figure paths to embed below:

In [ ]:
# TODO: display a Task I plot
# from IPython.display import Image, display
# display(Image(filename=TASK1_PLOTS / "packet/distribution/frame_len_pdf_control_vs_data.png"))

---
## Task II — P4 in-network feature extraction (brief recap)

**Pipeline:** `make run-grpc` → replay MAWI on h1 → capture telemetry on h2 → `run_pipeline.py`

**Switch:** BMv2 `simple_switch_grpc`, P4Runtime on port 50051, in-band `telemetry_t` header on egress.

**Flow key:** CRC16 hash of 5-tuple → 16 384 register slots (`p4_flow_id`).

### TODO: Task II run notes

- Packets replayed / captured: _?_
- Replay rate (`REPLAY_MULTIPLIER`): _?_
- BMv2 drops (if any):
- Register reset method: P4Runtime pipeline reload (`controller_grpc.py`)

In [ ]:
# TODO: display a Task II plot
# display(Image(filename=TASK2_PLOTS / "packet/distribution/frame_len_pdf_control_vs_data.png"))

---
## Task III — Comparison (main write-up)

Compare Task I (offline software) vs Task II (in-switch telemetry) on **comparable metrics** where possible.

### 3.1 Methodology differences

| Aspect | Task I | Task II |
|--------|--------|--------|
| Data path | Full MAWI JSONL / PCAP | h1 replay → P4 switch → h2 capture |
| Timestamps | Wireshark / trace time | BMv2 `ingress_global_timestamp` (µs) |
| Packet size | L2 `frame.len` | L3 length (IPv4 `totalLen` / IPv6 payload+40) |
| Flow ID | tcp/udp stream or 5-tuple | P4 hash mod 16384 |
| Truncation | MAWI ~96 B cap | Same on replay |

**TODO:** Expand with your pipeline commands and sample sizes.

### 3.2 Side-by-side metrics

**TODO:** Load fit summaries and compare best distributions / KS statistics.

In [ ]:
import pandas as pd

def load_fits(path: Path, label: str) -> pd.DataFrame:
    if not path.is_file():
        print(f"Missing {path} — run plot_analysis / run_pipeline first.")
        return pd.DataFrame()
    df = pd.read_csv(path)
    df["task"] = label
    return df

fits1 = load_fits(TASK1_FITS, "task1")
fits2 = load_fits(TASK2_FITS, "task2")

if not fits1.empty and not fits2.empty:
    cols = ["task", "dataset", "distribution", "n", "ks_statistic", "segment", "group"]
    display(pd.concat([fits1, fits2])[cols].query('segment == "all"'))

# TODO: comment on agreements / disagreements (IAT, frame_len, flow_byte_sum, ...)

### 3.3 Figure comparison

**TODO:** Place Task I vs Task II plots for the same metric (e.g. IAT PDF, flow size CCDF).

Discuss:
- Shape similarity or systematic shift
- Effect of L2 vs L3 length on frame/size distributions
- Effect of P4 hash collisions on flow-level stats

In [ ]:
# TODO: optional helper to show two images side by side
# from IPython.display import display, HTML
# display(HTML('<div style="display:flex;gap:1em">'
#   f'<img src="{TASK1_PLOTS}/..." width="45%">'
#   f'<img src="{TASK2_PLOTS}/..." width="45%">'
#   '</div>'))

### 3.4 Known limitations (required discussion)

**TODO:** Address each bullet with your evidence.

1. **Hash collisions** — 16k slots vs millions of flows; merged counters.
2. **Mice vs elephant split** — median `byte_sum`; document threshold used.
3. **MAWI truncation** — 96 B cap; TCP `tcp.len` vs payload semantics (Task I fix).
4. **BMv2 performance** — drops at high replay rate; actual capture count vs 10k target.
5. **P4Runtime register access** — BMv2 does not support register Read/Write over gRPC; telemetry via capture is the reliable path.
6. **Clock domain** — switch µs timestamps vs trace epoch in Task I.

### 3.5 Conclusion

**TODO:** 1–2 paragraphs.

- When does in-network extraction match software analysis?
- What breaks fidelity first (collisions, truncation, drops, …)?
- Would you trust this P4 program for operational measurement?

---
## AI use statement

**TODO (required by course):** Describe how you used AI tools (if any), which parts of the work they assisted with, and what you verified manually.

_Example structure:_
- Tools used:
- Assisted with:
- Not used for:
- Verification steps:

---
## Appendix — Reproduce commands

```bash
# Task I
cd task1 && python3 -m src.analysis.plot_analysis 201302011400.jsonl -n 10000

# Task II (Docker /p4)
make trace-prepare && make build && make run-grpc
sudo python3 control_plane/controller_grpc.py
REPLAY_MULTIPLIER=0.001 make test-mawi   # live sniff -> data/capture.jsonl
python3 control_plane/run_pipeline.py --capture-jsonl data/capture.jsonl

# Or plots + fits only from existing capture:
python3 control_plane/plot_analysis.py data/capture.jsonl
```